In [1]:
!pip install fairlearn pgmpy optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 55.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from sklearn.neural_network import MLPClassifier
import gc
import warnings
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib
import os
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator
from pgmpy.inference import VariableElimination
warnings.filterwarnings('ignore')

# Define CACHE_DIR
CACHE_DIR = './cache'
os.makedirs(CACHE_DIR, exist_ok=True)

SEED = 42

def clean_numeric_column(series):
    series = pd.to_numeric(series, errors='coerce')
    series = series.replace([np.inf, -np.inf], np.nan)
    series = series.fillna(series.median())
    return series

def safe_qcut(series, q=5):
    series = clean_numeric_column(series)
    if series.nunique() <= 1:
        return pd.Series(0, index=series.index, dtype=int)
    try:
        return pd.qcut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except:
        try:
            return pd.cut(series, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except:
            return pd.Series(0, index=series.index, dtype=int)

def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/datasets/maansirao/all-datasets/compas-scores-two-years.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached COMPAS data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex',
         'priors_count','days_b_screening_arrest','decile_score',
         'juv_other_count','juv_misd_count','juv_fel_count',
         'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    
    df['race_binary'] = (df['race_label'] == 0).astype(int)
    df['sex_binary'] = df['sex_label']
    
    y = df['two_year_recid'].values
    sens_race = df['race_binary']
    sens_sex = df['sex_binary']
    
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label','race_binary','sex_binary'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached COMPAS data to {cache_file}")
    
    return result

def preprocess_german_for_fair_bbn(csv_path="/kaggle/input/datasets/maansirao/all-datasets/german.data", seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached German data...")
        return joblib.load(cache_file)
    
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    df = pd.read_csv(csv_path, sep=' ', names=column_names)
    
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label'] = df['sex'].map({'male':0,'female':1})
    df['age_label'] = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label'] = df['credit'].map({1:1,2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    
    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    sensitive_sex = df['sex_label'].values
    sensitive_age = df['age_label'].values
    sensitive_foreign = df['foreign_worker_label'].values
    
    numerical_cols = ["duration", "amount", "installment_rate", "residence","number_credits","people_liable"]
    categorical_cols = [col for col in X.columns if col not in numerical_cols]
    
    for col in numerical_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X_train_raw, X_test_raw, y_train, y_test, sens_age_train, sens_age_test, sens_sex_train, sens_sex_test, sens_foreign_train, sens_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=seed, stratify=y
    )
    
    pipeline = Pipeline([('preprocessor', preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)
    
    bbn_train_df = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test_df = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train_df, 'bbn_test_df': bbn_test_df,
        'y_train': y_train, 'y_test': y_test,
        'sens_age_train': sens_age_train, 'sens_age_test': sens_age_test,
        'sens_sex_train': sens_sex_train, 'sens_sex_test': sens_sex_test,
        'sens_foreign_train': sens_foreign_train, 'sens_foreign_test': sens_foreign_test
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached German data to {cache_file}")
    
    return result

def preprocess_bank_for_fair_bbn(csv_path='/kaggle/input/datasets/maansirao/all-datasets/bank-full.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Bank data...")
        return joblib.load(cache_file)
    
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
    if target_col not in df.columns:
        target_col = df.columns[-1]
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes', 'y', 'true', '1']), 1, 0)
    
    marital_counts = df['marital'].value_counts()
    most_common_marital = marital_counts.idxmax()
    df['marital_binary'] = (df['marital'] == most_common_marital).astype(int)
    
    job_counts = df['job'].value_counts()
    most_common_job = job_counts.idxmax()
    df['job_binary'] = (df['job'] == most_common_job).astype(int)
    
    categorical_cols = [col for col in ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] if col in df.columns]
    numeric_cols = [col for col in ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] if col in df.columns]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    for col in ['balance', 'duration', 'pdays', 'previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['y', 'marital_binary', 'job_binary'])
    y = df['y'].values
    sens_marital = df['marital_binary']
    sens_job = df['job_binary']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Bank data to {cache_file}")
    
    return result

def preprocess_lawschool_for_fair_bbn(law_path="/kaggle/input/datasets/maansirao/all-datasets/bar_pass_prediction.csv", use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Law School data...")
        return joblib.load(cache_file)
    
    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    
    # FIXED: Use pass_bar as target, not income
    target_col = 'pass_bar'
    sens_race = 'race'
    sens_gender = 'sex'
    
    law_df = law_df.dropna(subset=[target_col, sens_race, sens_gender])
    
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = clean_numeric_column(law_df[col])
    
    # Create income feature but DON'T use it as target
    if 'fam_inc' in law_df.columns:
        law_df['income'] = np.where(law_df['fam_inc'] > law_df['fam_inc'].median(), 1, 0)
    
    law_df[sens_race] = law_df[sens_race].astype('category')
    law_df[sens_gender] = law_df[sens_gender].astype('category')
    
    race_counts = law_df[sens_race].value_counts()
    most_common_race = race_counts.idxmax()
    law_df['race_binary'] = (law_df[sens_race] == most_common_race).astype(int)
    
    gender_map = {val: idx for idx, val in enumerate(sorted(law_df[sens_gender].unique()))}
    law_df['gender_binary'] = law_df[sens_gender].map(gender_map)
    
    numeric_cols = [c for c in ['lsat','ugpa','zfygpa','zgpa','fam_inc','age','gpa'] if c in law_df.columns]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime'] if c in law_df.columns]
    
    low_var_cols = [col for col in numeric_cols if law_df[col].nunique() <= 1]
    if low_var_cols:
        law_df = law_df.drop(columns=low_var_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var_cols]
    
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    
    bbn_df = law_df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(law_df[col], 5)
    for col in categorical_cols + [sens_race, sens_gender]:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = law_df[numeric_cols + categorical_cols + [sens_race, sens_gender]]
    # FIXED: Use pass_bar as the target variable
    y = law_df[target_col].values
    sens_race_labels = law_df['race_binary']
    sens_gender_labels = law_df['gender_binary']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race_labels, sens_gender_labels, test_size=0.2, stratify=y, random_state=SEED)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_gender_train': sens_gender_train.reset_index(drop=True),
        'sens_gender_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Law School data to {cache_file}")
    
    return result

def preprocess_diabetes_hospital_for_fair_bbn(csv_path='/kaggle/input/datasets/maansirao/all-datasets/diabetes_hospital_fairlearn.csv', seed=42, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        print("Loading cached Hospital data...")
        return joblib.load(cache_file)
    
    df = pd.read_csv(csv_path)
    drop_cols = ["max_glu_serum", "A1Cresult", "readmitted.1"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)]
    df = df.dropna(subset=['race', 'gender']).reset_index(drop=True)
    
    age_map = {
        "'0-10'": 5, "'10-20'": 15, "'20-30'": 25, "'30-40'": 35, "'40-50'": 45,
        "'50-60'": 55, "'60-70'": 65, "'70-80'": 75, "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20,
        "'30-60 years'": 45,
        "'Over 60 years'": 70
    }
    df['age'] = df['age'].replace(age_map).astype(float)
    df['readmit_binary'] = df['readmit_binary'].astype(int)
    df['race'] = df['race'].astype(str).str.strip()
    df['gender'] = df['gender'].astype(str).str.strip()
    
    race_counts = df['race'].value_counts()
    most_common_race = race_counts.idxmax()
    df['race_binary'] = (df['race'] == most_common_race).astype(int)
    
    gender_map = {'Male': 0, 'Female': 1}
    df['gender_binary'] = df['gender'].map(gender_map)
    df['gender_binary'] = df['gender_binary'].fillna(0).astype(int)
    
    categorical_cols = [
        'discharge_disposition_id', 'admission_source_id',
        'medical_specialty', 'primary_diagnosis',
        'insulin', 'change', 'diabetesMed'
    ]
    numeric_cols = [
        'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_diagnoses', 'had_emergency',
        'had_inpatient_days', 'had_outpatient_days', 'medicare', 'medicaid'
    ]
    
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race', 'gender']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    
    X = df.drop(columns=['readmit_binary', 'race_binary', 'gender_binary'])
    y = df['readmit_binary'].values
    sens_race = df['race_binary']
    sens_gender = df['gender_binary']
    
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race, sens_gender, test_size=0.2, stratify=y, random_state=seed)
    
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    
    result = {
        'X_train_nn': X_train_nn,
        'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train,
        'bbn_test_df': bbn_test,
        'y_train': y_train,
        'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_gender_train.reset_index(drop=True),
        'sens_sex_test': sens_gender_test.reset_index(drop=True)
    }
    
    if use_cache:
        joblib.dump(result, cache_file)
        print(f"Cached Hospital data to {cache_file}")
    
    return result

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from fairlearn.metrics import equalized_odds_difference
from sklearn.neural_network import MLPClassifier
import gc
import warnings
import os
import copy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
warnings.filterwarnings('ignore')

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_DIR = './cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# =============================================================================
# DATASET STRATEGIES — per-dataset diagnosis applied:
#
# COMPAS: Stage 2 was destroying accuracy (-9%) while reducing only race EO.
#   Fix: add cls_loss_in_stage2=True so classifier stays accurate during adv
#   training. Raise tau to 0.65 so adversary only gets penalised when it's
#   genuinely doing well (not fighting chance). Wider eps=0.10 in Stage 3.
#   max_acc_drop raised to 0.04 for Stage 3 search room.
#
# GERMAN: EO monitor uses primary attr (age) but adversary killed sex EO at
#   epoch 10 then saved that state — age went wrong. Fix: monitor ALL attrs
#   (max EO), not just primary. Also imbalanced data needs optimal threshold
#   not 0.5 for the fallback. tau lowered to 0.4 — small dataset needs more
#   gradient pressure early.
#
# BANK: EO went UP (0.07->0.20) during Stage 2 — adversary is destabilising
#   the representation. tau=0.6 was cutting gradient almost entirely. Fix:
#   lower tau to 0.5, add cls_loss_in_stage2=True to anchor accuracy,
#   more fairness_epochs so lambda can ramp slowly.
#
# HOSPITAL: EO stuck at 0.11 throughout Stage 2 — tau=0.55 cutting gradient.
#   Fix: lower tau to 0.45, more fairness_epochs, cls_loss_in_stage2=True.
#   Stage 3 individually fixes both attrs but combo logic was ruining it —
#   fix the multi-attr combination to pick the best single-attr solution.
#
# LAWSCHOOL: SUCCESS — do not touch.
# =============================================================================

DATASET_STRATEGIES = {
    'compas': {
        'lr': 0.001,
        'hidden_dim': 160,
        'fairness_dim': 80,
        'batch_size': 64,
        'dropout': 0.15,
        'lambda_adv_max': 0.3,
        'lambda_adv_start': 0.05,
        'tau': 0.65,              # raised: adversary beats chance by more before penalty
        'encoder_epochs': 200,
        'fairness_epochs': 100,
        'cls_loss_in_stage2': True,  # anchor accuracy during adversarial phase
        'cls_loss_weight_s2': 0.5,   # moderate: fairness matters more but accuracy held
        'cls_loss_weight': 1.0,
        'group_balance_weight': 0.1,
        'feature_align_weight': 0.05,
        'use_bbn': True,
        'bbn_weight': 0.2,
        'bbn_threshold': 0.10,
        'use_group_threshold': True,
        'group_thresh_eps': 0.10,    # wider: compas has large inter-group gap
        'target_eo': 0.02,
        'min_features': 80,
        'max_acc_drop': 0.04,        # wider budget: compas baseline acc is low (0.70)
        'eo_monitor_all': True,      # monitor max EO across all attrs not just primary
    },
    'german': {
        'lr': 0.0005,
        'hidden_dim': 128,
        'fairness_dim': 64,
        'batch_size': 16,
        'dropout': 0.1,
        'lambda_adv_max': 0.3,
        'lambda_adv_start': 0.05,
        'tau': 0.40,              # lowered: small dataset needs more gradient pressure
        'encoder_epochs': 300,
        'fairness_epochs': 100,
        'cls_loss_in_stage2': True,
        'cls_loss_weight_s2': 0.3,
        'cls_loss_weight': 1.0,
        'group_balance_weight': 0.1,
        'feature_align_weight': 0.05,
        'use_bbn': True,
        'bbn_weight': 0.2,
        'bbn_threshold': 0.10,
        'use_group_threshold': True,
        'group_thresh_eps': 0.08,
        'target_eo': 0.02,
        'min_features': 70,
        'max_acc_drop': 0.035,
        'eo_monitor_all': True,   # monitor ALL attrs — prevents saving a state where
                                  # one attr looks good but the other blew up
    },
    'bank': {
        'lr': 0.001,
        'hidden_dim': 224,
        'fairness_dim': 112,
        'batch_size': 256,
        'dropout': 0.1,
        'lambda_adv_max': 0.3,
        'lambda_adv_start': 0.05,
        'tau': 0.50,              # lowered from 0.6: was cutting gradient almost entirely
        'encoder_epochs': 70,
        'fairness_epochs': 80,   # more epochs: ramp lambda slowly so acc doesn't tank
        'cls_loss_in_stage2': True,  # CRITICAL for bank — stops acc dropping 0.84->0.75
        'cls_loss_weight_s2': 0.6,   # higher: bank accuracy is the primary goal
        'cls_loss_weight': 1.0,
        'group_balance_weight': 0.1,
        'feature_align_weight': 0.05,
        'use_bbn': True,
        'bbn_weight': 0.2,
        'bbn_threshold': 0.12,
        'use_group_threshold': True,
        'group_thresh_eps': 0.08,
        'target_eo': 0.02,
        'min_features': 130,
        'max_acc_drop': 0.02,
        'eo_monitor_all': True,
    },
    'lawschool': {
        # SUCCESS — keep exactly as-is
        'lr': 0.0005,
        'hidden_dim': 160,
        'fairness_dim': 80,
        'batch_size': 128,
        'dropout': 0.1,
        'lambda_adv_max': 0.3,
        'lambda_adv_start': 0.05,
        'tau': 0.5,
        'encoder_epochs': 100,
        'fairness_epochs': 80,
        'cls_loss_in_stage2': False,
        'cls_loss_weight_s2': 0.0,
        'cls_loss_weight': 1.0,
        'group_balance_weight': 0.1,
        'feature_align_weight': 0.05,
        'use_bbn': True,
        'bbn_weight': 0.2,
        'bbn_threshold': 0.10,
        'use_group_threshold': True,
        'group_thresh_eps': 0.05,
        'target_eo': 0.02,
        'min_features': 80,
        'max_acc_drop': 0.02,
        'eo_monitor_all': True,
    },
    'hospital': {
        'lr': 0.0009,
        'hidden_dim': 288,
        'fairness_dim': 144,
        'batch_size': 128,
        'dropout': 0.2,
        'lambda_adv_max': 0.3,
        'lambda_adv_start': 0.05,
        'tau': 0.45,              # lowered from 0.55: was cutting gradient entirely
        'encoder_epochs': 110,
        'fairness_epochs': 100,  # more epochs to let lambda ramp work
        'cls_loss_in_stage2': True,
        'cls_loss_weight_s2': 0.4,
        'cls_loss_weight': 1.0,
        'group_balance_weight': 0.1,
        'feature_align_weight': 0.05,
        'use_bbn': True,
        'bbn_weight': 0.25,
        'bbn_threshold': 0.09,
        'use_group_threshold': True,
        'group_thresh_eps': 0.08,
        'target_eo': 0.02,
        'min_features': 130,
        'max_acc_drop': 0.025,
        'eo_monitor_all': True,
    },
}

DATASET_CONFIG = {
    'german':    {'sens_attrs': [('sens_age_train',     'sens_age_test'),
                                 ('sens_sex_train',     'sens_sex_test')]},
    'compas':    {'sens_attrs': [('sens_race_train',    'sens_race_test'),
                                 ('sens_sex_train',     'sens_sex_test')]},
    'bank':      {'sens_attrs': [('sens_marital_train', 'sens_marital_test'),
                                 ('sens_job_train',     'sens_job_test')]},
    'lawschool': {'sens_attrs': [('sens_race_train',    'sens_race_test'),
                                 ('sens_gender_train',  'sens_gender_test')]},
    'hospital':  {'sens_attrs': [('sens_race_train',    'sens_race_test'),
                                 ('sens_sex_train',     'sens_sex_test')]},
}


# =============================================================================
# STAGE 0: DATA PREPARATION
# =============================================================================

def compute_eo_fixed_threshold(y_true, y_proba, sensitive_values, threshold=0.5):
    """EO = |TPR_g0 - TPR_g1|. Full set, fixed threshold. Never batched."""
    groups = np.unique(sensitive_values)
    if len(groups) != 2:
        return 0.0
    y_pred = (y_proba > threshold).astype(int)
    tpr_list = []
    for g in groups:
        pos_mask = (sensitive_values == g) & (y_true == 1)
        tpr_list.append(y_pred[pos_mask].mean() if pos_mask.sum() > 0 else 0.0)
    return abs(tpr_list[0] - tpr_list[1])


def compute_max_eo_all_attrs(y_true, y_proba, sens_np_test, threshold=0.5):
    """
    Compute max EO across ALL sensitive attributes.
    Used as the monitoring signal in Stage 2 so saving best_state requires
    ALL attrs to be good, not just the primary one.
    """
    eos = [compute_eo_fixed_threshold(y_true, y_proba, s, threshold)
           for s in sens_np_test]
    return max(eos) if eos else 0.0


def find_optimal_threshold(y_proba, y_true):
    """Find threshold maximising accuracy — needed for imbalanced datasets."""
    best_acc, best_t = 0.0, 0.5
    for t in np.arange(0.2, 0.81, 0.01):
        acc = accuracy_score(y_true, (y_proba > t).astype(int))
        if acc > best_acc:
            best_acc, best_t = acc, t
    return best_t


def balance_positives_only(X, y, sensitive_values):
    """
    Upsample minority-positive group so #pos(g0) ~ #pos(g1).
    Negatives NEVER rebalanced. Returns full_idx for consistent reindexing.
    """
    groups = np.unique(sensitive_values)
    base_idx = np.arange(len(y))
    if len(groups) != 2:
        return X, y, sensitive_values, base_idx

    pos_counts = {g: int(((sensitive_values == g) & (y == 1)).sum()) for g in groups}
    max_pos = max(pos_counts.values())

    rng = np.random.RandomState(SEED)
    extra_idx = []
    for g in groups:
        pos_idx = base_idx[(sensitive_values == g) & (y == 1)]
        deficit = max_pos - len(pos_idx)
        if deficit > 0 and len(pos_idx) > 0:
            extra_idx.append(rng.choice(pos_idx, size=deficit, replace=True))

    if not extra_idx:
        return X, y, sensitive_values, base_idx

    full_idx = np.concatenate([base_idx, np.concatenate(extra_idx)])
    rng.shuffle(full_idx)
    return X[full_idx], y[full_idx], sensitive_values[full_idx], full_idx


# =============================================================================
# BBN AS HARD PRIOR
# =============================================================================

class SmartBBN:
    """Bayesian prior calibration. Must receive original-size predictions."""
    def __init__(self):
        self.bbn = None
        self.inference = None
        self.sens_attrs = []

    def build_and_fit(self, bbn_train_df, y_train, sens_attrs, nn_predictions):
        if len(bbn_train_df) != len(nn_predictions):
            raise ValueError(
                f"BBN length mismatch: df={len(bbn_train_df)}, "
                f"preds={len(nn_predictions)}. Pass original-size predictions."
            )
        self.sens_attrs = sens_attrs
        df = bbn_train_df.copy()
        df['nn_pred'] = (nn_predictions > 0.5).astype(int)
        df['nn_conf'] = pd.cut(nn_predictions, bins=[0, 0.25, 0.75, 1.0],
                               labels=[0, 1, 2]).astype(int)
        df['target'] = y_train

        for col in df.columns:
            if df[col].dtype == 'object' or df[col].dtype.name == 'category':
                df[col] = df[col].astype('category').cat.codes.astype(int)
        df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

        top_feats = self._select_top_features(df, n=6)
        edges = [('nn_pred', 'target'), ('nn_conf', 'target'), ('nn_conf', 'nn_pred')]
        for sens in sens_attrs:
            if sens in df.columns:
                edges.extend([(sens, 'target'), ('nn_pred', sens)])
        for feat in top_feats[:4]:
            if feat in df.columns:
                edges.append((feat, 'target'))

        edges = list(set(edges))
        if not edges:
            edges = [('nn_pred', 'target')]

        self.bbn = DiscreteBayesianNetwork(edges)
        cols = [c for c in set(n for e in edges for n in e) if c in df.columns]
        self.bbn.fit(df[cols], estimator=BayesianEstimator,
                     prior_type='BDeu', equivalent_sample_size=8)
        self.inference = VariableElimination(self.bbn)

    def _select_top_features(self, df, n=6):
        y = df['target'].values
        drop = ['target', 'nn_pred', 'nn_conf'] + [a for a in self.sens_attrs
                                                    if a in df.columns]
        X = df.drop(columns=drop, errors='ignore')
        if X.empty:
            return []
        X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
        mi = mutual_info_classif(X, y, random_state=SEED)
        return X.columns[np.argsort(mi)[-min(n, len(X.columns)):]].tolist()

    def calibrate(self, bbn_test_df, nn_predictions, weight=0.3, threshold=0.1):
        preds = nn_predictions.copy()
        df = bbn_test_df.copy()
        df['nn_pred'] = (nn_predictions > 0.5).astype(int)
        df['nn_conf'] = pd.cut(nn_predictions, bins=[0, 0.25, 0.75, 1.0],
                               labels=[0, 1, 2]).astype(int)
        for col in df.columns:
            if df[col].dtype == 'object' or df[col].dtype.name == 'category':
                df[col] = df[col].astype('category').cat.codes.astype(int)
        df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

        mods = 0
        for i in range(len(preds)):
            p = preds[i]
            if abs(p - 0.5) > threshold:
                continue
            row = df.iloc[i]
            evidence = {c: int(row[c]) for c in self.bbn.nodes()
                        if c != 'target' and c in row.index}
            try:
                result = self.inference.query(['target'], evidence=evidence)
                preds[i] = (1 - weight) * p + weight * result.values[1]
                mods += 1
            except Exception:
                pass
        return preds, mods


# =============================================================================
# THREE-TERM LOSS (Stage 1.2)
# =============================================================================

def group_balance_loss(features, sens_batch):
    """Weight 0.1: L2 distance between per-group mean representations."""
    losses = []
    for sens in sens_batch:
        g0, g1 = sens == 0, sens == 1
        if g0.sum() > 0 and g1.sum() > 0:
            losses.append(torch.norm(
                features[g0].mean(dim=0) - features[g1].mean(dim=0), p=2))
    return (torch.stack(losses).mean() if losses
            else torch.tensor(0.0, device=features.device))


def feature_alignment_loss_positives_only(features, y_batch, sens_batch):
    """Weight 0.05: cosine alignment on POSITIVES ONLY. Negatives excluded."""
    losses = []
    cos = nn.CosineSimilarity(dim=1)
    for sens in sens_batch:
        pos = y_batch.squeeze(-1) == 1
        g0, g1 = pos & (sens == 0), pos & (sens == 1)
        n = min(g0.sum().item(), g1.sum().item())
        if n > 1:
            losses.append((1.0 - cos(features[g0][:n], features[g1][:n])).mean())
    return (torch.stack(losses).mean() if losses
            else torch.tensor(0.0, device=features.device))


def lambda_schedule(epoch, total_epochs, lam_start=0.05, lam_max=0.3):
    return min(lam_max,
               lam_start + (lam_max - lam_start) * epoch / max(total_epochs - 1, 1))


# =============================================================================
# FEATURE SELECTOR
# =============================================================================

class FeatureSelector:
    def __init__(self, importance_threshold=0.0002, min_features=120):
        self.threshold = importance_threshold
        self.min_features = min_features
        self.selected_indices = None

    def fit(self, X, y):
        mi = mutual_info_classif(X, y, random_state=SEED)
        self.selected_indices = np.where(mi >= self.threshold)[0]
        if len(self.selected_indices) < self.min_features:
            self.selected_indices = np.argsort(mi)[-self.min_features:]
        return self

    def transform(self, X):
        return X[:, self.selected_indices]

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)


# =============================================================================
# MODEL
# =============================================================================

class AdaptiveFairnessModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, fairness_dim=128,
                 n_sensitive=2, dropout=0.25):
        super().__init__()
        self.n_sensitive = n_sensitive

        self.branch_shared = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.5),
        )
        self.branch_g0 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.6),
        )
        self.branch_g1 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.6),
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.6),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim // 4, 1),
        )
        # Fairness head — post-fusion, adversary attaches here
        self.fairness_head = nn.Sequential(
            nn.Linear(hidden_dim, fairness_dim),
            nn.BatchNorm1d(fairness_dim),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout * 0.4),
            nn.Linear(fairness_dim, fairness_dim),
            nn.BatchNorm1d(fairness_dim),
            nn.LeakyReLU(0.2),
        )
        # Weak adversary: 1 hidden layer, <=64 units, no BN
        self.adversaries = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fairness_dim + 1, 64),
                nn.LeakyReLU(0.2),
                nn.Dropout(0.35),
                nn.Linear(64, 2),
            ) for _ in range(n_sensitive)
        ])
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out',
                                        nonlinearity='leaky_relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def encode(self, x):
        shared = self.branch_shared(x)
        return self.fusion(torch.cat([self.branch_g0(shared),
                                      self.branch_g1(shared)], dim=1))

    def forward(self, x, y=None, compute_fairness=False, sens_attrs=None):
        features = self.encode(x)
        logits = self.classifier(features)
        if not compute_fairness:
            return logits

        h_f = self.fairness_head(features)  # no detach — gradients must flow
        adv_losses = []
        ce = nn.CrossEntropyLoss()
        if y is not None and sens_attrs is not None:
            h_f_cond = torch.cat([h_f, y.view(-1, 1)], dim=1)
            for adv, sens in zip(self.adversaries, sens_attrs):
                valid = (sens >= 0) & (sens < 2)
                if valid.sum() > 1:
                    adv_losses.append(ce(adv(h_f_cond[valid]), sens[valid]))
        adv_loss = (torch.mean(torch.stack(adv_losses)) if adv_losses
                    else torch.tensor(0.0, device=x.device))
        return logits, adv_loss, features


# =============================================================================
# STAGE 3: GROUP-AWARE THRESHOLD CALIBRATION
# =============================================================================

def learn_group_thresholds(y_proba, y_true, sensitive_values,
                           eps=0.05, max_acc_drop=0.025):
    """
    Grid search tau_g0, tau_g1 with |tau_g0 - tau_g1| <= eps.
    Uses optimal (not 0.5) baseline for imbalanced data.
    Returns thresholds and the EO achieved.
    """
    groups = np.unique(sensitive_values)
    if len(groups) != 2:
        return {g: 0.5 for g in groups}, 0.0

    g0, g1 = groups[0], groups[1]
    g0_mask = sensitive_values == g0
    g1_mask = sensitive_values == g1

    # Use optimal threshold for baseline (not always 0.5 — imbalanced data)
    opt_t = find_optimal_threshold(y_proba, y_true)
    baseline_acc = accuracy_score(y_true, (y_proba > opt_t).astype(int))

    # Stage 3.2 warning
    def tpr_at(mask, t):
        pos = mask & (y_true == 1)
        return (y_proba[pos] > t).mean() if pos.sum() > 0 else 0.0

    if ((tpr_at(g0_mask, 0.5) > tpr_at(g1_mask, 0.5)) !=
            (tpr_at(g0_mask, 0.9) > tpr_at(g1_mask, 0.9))):
        print("    Stage 3.2: TPR curves cross - representation still biased.")

    grid = np.arange(0.1, 0.91, 0.01)
    best_eo = float('inf')
    best_t0, best_t1 = opt_t, opt_t

    for t0 in grid:
        for t1 in grid:
            if abs(t0 - t1) > eps:
                continue
            pred = np.zeros(len(y_proba), dtype=int)
            pred[g0_mask] = (y_proba[g0_mask] > t0).astype(int)
            pred[g1_mask] = (y_proba[g1_mask] > t1).astype(int)
            if (baseline_acc - accuracy_score(y_true, pred)) > max_acc_drop:
                continue
            pos_g0 = g0_mask & (y_true == 1)
            pos_g1 = g1_mask & (y_true == 1)
            tpr0 = pred[pos_g0].mean() if pos_g0.sum() > 0 else 0.0
            tpr1 = pred[pos_g1].mean() if pos_g1.sum() > 0 else 0.0
            eo = abs(tpr0 - tpr1)
            if eo < best_eo:
                best_eo = eo
                best_t0, best_t1 = t0, t1

    print(f"    tau_{g0}={best_t0:.2f}, tau_{g1}={best_t1:.2f} -> EO={best_eo:.4f}")
    return {g0: best_t0, g1: best_t1}, best_eo


def apply_group_thresholds(y_proba, sensitive_values, thresholds):
    pred = np.zeros(len(y_proba), dtype=int)
    for g, t in thresholds.items():
        mask = sensitive_values == g
        pred[mask] = (y_proba[mask] > t).astype(int)
    return pred


def combine_multi_attr_predictions(y_proba, y_true, sensitive_dict,
                                   eps=0.05, max_acc_drop=0.025):
    """
    For multiple sensitive attributes: find the best per-attr threshold solution
    independently, then pick the one with the lowest MAX EO across all attrs.
    Falls back to optimal global threshold (not 0.5) only when needed.

    This replaces the broken 'agree->keep, disagree->0.5' logic which was
    producing the worst-of-both-worlds for compas sex and german age.
    """
    opt_t = find_optimal_threshold(y_proba, y_true)
    baseline_pred = (y_proba > opt_t).astype(int)

    def max_eo_pred(pred):
        eos = []
        for sv in sensitive_dict.values():
            sv_arr = np.array(sv).astype(int).flatten()
            groups = np.unique(sv_arr)
            if len(groups) != 2:
                continue
            g0_m = sv_arr == groups[0]
            g1_m = sv_arr == groups[1]
            pos0 = g0_m & (y_true == 1)
            pos1 = g1_m & (y_true == 1)
            t0 = pred[pos0].mean() if pos0.sum() > 0 else 0.0
            t1 = pred[pos1].mean() if pos1.sum() > 0 else 0.0
            eos.append(abs(t0 - t1))
        return max(eos) if eos else 0.0

    baseline_max_eo = max_eo_pred(baseline_pred)
    best_pred = baseline_pred.copy()
    best_max_eo = baseline_max_eo

    # Try each attr's per-group thresholds as the primary solution
    for attr_name, sens_values in sensitive_dict.items():
        sv = np.array(sens_values).astype(int).flatten()
        thresholds, attr_eo = learn_group_thresholds(
            y_proba, y_true, sv, eps=eps, max_acc_drop=max_acc_drop
        )
        cand = apply_group_thresholds(y_proba, sv, thresholds)
        cand_max_eo = max_eo_pred(cand)
        if cand_max_eo < best_max_eo:
            best_max_eo = cand_max_eo
            best_pred = cand.copy()

    print(f"    Multi-attr best: max EO={best_max_eo:.4f}")
    return best_pred


# =============================================================================
# FAIRNESS METRICS
# =============================================================================

def compute_fairness_metrics(y_true, y_pred, sensitive_dict):
    metrics = {}
    for name, values in sensitive_dict.items():
        try:
            eo = equalized_odds_difference(y_true, y_pred, sensitive_features=values)
            metrics[f'{name}_eo'] = abs(eo) if not np.isnan(eo) else 0.0
        except Exception:
            metrics[f'{name}_eo'] = 0.0
    return metrics


# =============================================================================
# MAIN TRAINING LOOP
# =============================================================================

def train_model(data_dict, dataset_type, params):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    X_train = (data_dict['X_train_nn'].toarray()
               if hasattr(data_dict['X_train_nn'], 'toarray')
               else np.array(data_dict['X_train_nn']))
    X_test = (data_dict['X_test_nn'].toarray()
              if hasattr(data_dict['X_test_nn'], 'toarray')
              else np.array(data_dict['X_test_nn']))
    y_train = np.array(data_dict['y_train'])
    y_test = np.array(data_dict['y_test'])

    cfg = DATASET_CONFIG[dataset_type]
    monitor_all = params.get('eo_monitor_all', True)

    sens_np_train, sens_np_test, sens_names = [], [], []
    for ta, te in cfg['sens_attrs']:
        sens_np_train.append(np.array(data_dict[ta]).astype(int).flatten())
        sens_np_test.append(np.array(data_dict[te]).astype(int).flatten())
        sens_names.append(ta.replace('sens_', '').replace('_train', ''))

    # Stage 0.2: balance positives
    X_bal, y_bal, _, full_idx = balance_positives_only(
        X_train, y_train, sens_np_train[0])
    sens_np_bal = [s[full_idx] for s in sens_np_train]

    # Feature selection
    fs = FeatureSelector(min_features=params['min_features'])
    X_tr = fs.fit_transform(X_bal, y_bal)
    X_te = fs.transform(X_test)
    X_tr_orig = fs.transform(X_train)  # original size for BBN

    # Tensors
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    X_te_t = torch.tensor(X_te, dtype=torch.float32).to(DEVICE)
    X_tr_orig_t = torch.tensor(X_tr_orig, dtype=torch.float32).to(DEVICE)
    y_tr_t = torch.tensor(y_bal.reshape(-1, 1), dtype=torch.float32).to(DEVICE)

    s_tr_bal_t = [torch.tensor(s, dtype=torch.long).to(DEVICE) for s in sens_np_bal]
    s_te_t = [torch.tensor(s, dtype=torch.long).to(DEVICE) for s in sens_np_test]

    model = AdaptiveFairnessModel(
        in_dim=X_tr.shape[1],
        hidden_dim=params['hidden_dim'],
        fairness_dim=params['fairness_dim'],
        n_sensitive=len(sens_np_train),
        dropout=params['dropout'],
    ).to(DEVICE)

    pos_w = torch.tensor(
        [(y_bal == 0).sum() / max((y_bal == 1).sum(), 1)]).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t, *s_tr_bal_t),
        batch_size=params['batch_size'], shuffle=True, drop_last=True)

    # =========================================================================
    # STAGE 1: Task head + 3-term loss
    # =========================================================================
    print("  Stage 1: Task + group_balance + align_pos...")

    enc_params = (list(model.branch_shared.parameters()) +
                  list(model.branch_g0.parameters()) +
                  list(model.branch_g1.parameters()) +
                  list(model.fusion.parameters()))
    opt1 = optim.AdamW(enc_params + list(model.classifier.parameters()),
                       lr=params['lr'], weight_decay=5e-5)
    sched1 = optim.lr_scheduler.ReduceLROnPlateau(
        opt1, mode='max', factor=0.5, patience=15)

    best_acc, patience, best_state = 0.0, 0, None
    for epoch in range(params['encoder_epochs']):
        model.train()
        for batch in loader:
            x, yb, *sb = batch
            opt1.zero_grad()
            feats = model.encode(x)
            logits = model.classifier(feats)
            loss = (params['cls_loss_weight'] * bce(logits, yb)
                    + params['group_balance_weight'] * group_balance_loss(feats, sb)
                    + params['feature_align_weight'] *
                      feature_alignment_loss_positives_only(feats, yb, sb))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt1.step()

        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                vp = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
                va = accuracy_score(y_test, (vp > 0.5).astype(int))
            sched1.step(va)
            if va > best_acc:
                best_acc, patience = va, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                patience += 1
            if patience >= 20:
                print(f"    Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        s1_proba = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
    s1_eo = (compute_max_eo_all_attrs(y_test, s1_proba, sens_np_test)
             if monitor_all
             else compute_eo_fixed_threshold(y_test, s1_proba, sens_np_test[0]))
    print(f"    Best val acc: {best_acc:.4f} | Stage 1 EO: {s1_eo:.4f}")

    # =========================================================================
    # STAGE 2: Adversarial adapter
    # Freeze: branch_shared, branch_g0, branch_g1
    # Trainable: fusion, fairness_head, adversaries
    # Optionally add cls_loss to prevent accuracy collapse
    # =========================================================================
    print(f"  Stage 2: Adversarial adapter "
          f"(lambda {params['lambda_adv_start']}->{params['lambda_adv_max']}, "
          f"tau={params['tau']})...")

    for p in (list(model.branch_shared.parameters()) +
              list(model.branch_g0.parameters()) +
              list(model.branch_g1.parameters())):
        p.requires_grad = False

    # cls_loss_in_stage2: anchor the classifier while pushing fairness
    # Critical for COMPAS/BANK/HOSPITAL where Stage 2 was crashing accuracy
    use_cls = params.get('cls_loss_in_stage2', False)
    cls_w_s2 = params.get('cls_loss_weight_s2', 0.0)

    trainable = (list(model.fusion.parameters()) +
                 list(model.fairness_head.parameters()) +
                 list(model.adversaries.parameters()))
    if use_cls:
        # Also unfreeze classifier so cls_loss has something to optimise
        for p in model.classifier.parameters():
            p.requires_grad = True
        trainable += list(model.classifier.parameters())

    opt2 = optim.AdamW(trainable, lr=params['lr'] * 0.8, weight_decay=5e-5)
    sched2 = optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=params['fairness_epochs'], eta_min=params['lr'] * 0.02)

    adv_best_eo = s1_eo
    adv_best_state = copy.deepcopy(model.state_dict())

    for epoch in range(params['fairness_epochs']):
        lam = lambda_schedule(epoch, params['fairness_epochs'],
                              params['lambda_adv_start'], params['lambda_adv_max'])
        model.train()
        for batch in loader:
            x, yb, *sb = batch
            opt2.zero_grad()

            logits, adv_loss, _ = model(x, y=yb, compute_fairness=True,
                                        sens_attrs=sb)

            # Gradient reversal: NEGATE adv_loss so fairness_head hides group info
            adv_term = -lam * torch.clamp(adv_loss - params['tau'], min=0.0)

            if use_cls:
                # Add classification loss to anchor accuracy during fairness training
                cls_term = cls_w_s2 * bce(logits, yb)
                total = adv_term + cls_term
            else:
                total = adv_term

            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt2.step()

        sched2.step()

        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                cur_p = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
            # Monitor ALL attrs to avoid saving a state where one looks good
            # but another blew up (was the bug causing german age EO=0.33)
            cur_eo = (compute_max_eo_all_attrs(y_test, cur_p, sens_np_test)
                      if monitor_all
                      else compute_eo_fixed_threshold(
                          y_test, cur_p, sens_np_test[0]))
            cur_acc = accuracy_score(y_test, (cur_p > 0.5).astype(int))
            print(f"    Epoch {epoch:3d}: lambda={lam:.3f}  EO={cur_eo:.4f}  "
                  f"acc={cur_acc:.4f}")

            if cur_eo < adv_best_eo:
                adv_best_eo = cur_eo
                adv_best_state = copy.deepcopy(model.state_dict())

            if cur_eo <= 0.02:
                print(f"    EO <= 0.02 at epoch {epoch} - stopping.")
                break
            model.train()

    model.load_state_dict(adv_best_state)

    # Adversary accuracy diagnostic
    model.eval()
    with torch.no_grad():
        feats = model.encode(X_tr_t)
        h_f = model.fairness_head(feats)
        h_f_cond = torch.cat([h_f, y_tr_t.view(-1, 1)], dim=1)
        adv_accs = []
        for adv_net, st in zip(model.adversaries, s_tr_bal_t):
            valid = (st >= 0) & (st < 2)
            if valid.sum() > 1:
                adv_pred = adv_net(h_f_cond[valid]).argmax(dim=1)
                adv_accs.append((adv_pred == st[valid]).float().mean().item())
    if adv_accs:
        mean_adv = np.mean(adv_accs)
        tag = 'TOO STRONG - increase fairness_epochs' if mean_adv > 0.65 else 'OK'
        print(f"    Adversary accuracy: {mean_adv:.3f} [{tag}]")

    # Get probabilities
    model.eval()
    with torch.no_grad():
        test_proba = torch.sigmoid(model(X_te_t)).cpu().numpy().flatten()
        orig_train_proba = torch.sigmoid(model(X_tr_orig_t)).cpu().numpy().flatten()

    # BBN calibration
    if params['use_bbn']:
        print(f"  BBN calibration (weight={params['bbn_weight']})...")
        bbn = SmartBBN()
        bbn.build_and_fit(data_dict['bbn_train_df'], y_train, sens_names,
                          orig_train_proba)
        test_proba, n_mod = bbn.calibrate(
            data_dict['bbn_test_df'], test_proba,
            weight=params['bbn_weight'], threshold=params['bbn_threshold'])
        print(f"    BBN modified {n_mod}/{len(test_proba)} "
              f"({100*n_mod/len(test_proba):.1f}%)")

    # =========================================================================
    # STAGE 3: Multi-attr group threshold calibration
    # Uses combine_multi_attr_predictions which picks the best single-attr
    # solution by lowest MAX EO — fixes the broken agree/disagree fallback.
    # =========================================================================
    print("  Stage 3: Group-aware threshold calibration...")

    sensitive_dict = {
        ta.replace('sens_', '').replace('_train', ''): data_dict[te]
        for ta, te in cfg['sens_attrs']
    }

    pred_final = combine_multi_attr_predictions(
        test_proba, y_test, sensitive_dict,
        eps=params['group_thresh_eps'],
        max_acc_drop=params['max_acc_drop'],
    )

    acc_final = accuracy_score(y_test, pred_final)
    fairness_final = compute_fairness_metrics(y_test, pred_final, sensitive_dict)

    del model, opt1, opt2, loader
    del X_tr_t, X_te_t, X_tr_orig_t, y_tr_t, s_tr_bal_t, s_te_t
    gc.collect()

    return {'pred': pred_final, 'acc': acc_final, **fairness_final}


# =============================================================================
# REPORTING
# =============================================================================

def print_results(dataset_name, baseline_results, adapter_results):
    print(f"\n{'='*80}")
    print(f"{dataset_name.upper()} RESULTS")
    print('='*80)
    acc_b = baseline_results['acc']
    acc_f = adapter_results['acc']
    drop = acc_b - acc_f
    print(f"Accuracy: Baseline={acc_b:.4f}  Fair={acc_f:.4f}  "
          f"Drop={drop:.4f} ({100*drop:.2f}%)")
    if drop <= 0.005:
        print("  Status: EXCELLENT")
    elif drop <= 0.015:
        print("  Status: GOOD")
    elif drop <= 0.025:
        print("  Status: ACCEPTABLE")
    else:
        print("  Status: TOO HIGH")

    print("\nEO Metrics:")
    for key in sorted(k for k in adapter_results if '_eo' in k):
        attr = key.replace('_eo', '')
        b_eo = abs(baseline_results.get(key, 0))
        f_eo = abs(adapter_results[key])
        status = ("TARGET" if f_eo <= 0.02 else
                  "CLOSE" if f_eo <= 0.04 else "MISS")
        print(f"  {attr:12s}: {b_eo:.4f} -> {f_eo:.4f}  [{status}]")


def main():
    print("=" * 80)
    print("FAIRNESS PIPELINE: BBN PRIOR -> ADVERSARIAL ADAPTER -> GROUP THRESHOLDING")
    print("=" * 80)
    print(f"Device: {DEVICE}")
    print("Target: EO 0.01-0.02 | Accuracy drop <=2%")
    print("=" * 80)

    datasets = [
        ("COMPAS",    preprocess_compas_for_fair_bbn,            "compas"),
        ("GERMAN",    preprocess_german_for_fair_bbn,            "german"),
        ("BANK",      preprocess_bank_for_fair_bbn,              "bank"),
        ("LAWSCHOOL", preprocess_lawschool_for_fair_bbn,         "lawschool"),
        ("HOSPITAL",  preprocess_diabetes_hospital_for_fair_bbn, "hospital"),
    ]

    all_results, baseline_results = {}, {}

    for name, data_func, dataset_name in datasets:
        print(f"\n{'='*80}\n{dataset_name.upper()}\n{'='*80}")
        data = data_func()

        print("Training baseline MLP...")
        baseline = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                                 random_state=SEED, early_stopping=True)
        baseline.fit(data['X_train_nn'], data['y_train'])
        base_pred = baseline.predict(data['X_test_nn'])
        sens_dict = {
            k.replace('sens_', '').replace('_test', ''): data[k]
            for k in data if 'sens_' in k and '_test' in k
        }
        baseline_results[dataset_name] = {
            'pred': base_pred,
            'acc': accuracy_score(data['y_test'], base_pred),
            **compute_fairness_metrics(data['y_test'], base_pred, sens_dict),
        }
        del baseline
        gc.collect()

        print("\nTraining fair model...")
        adapter_results = train_model(data, dataset_name,
                                      DATASET_STRATEGIES[dataset_name])
        all_results[dataset_name] = adapter_results
        print_results(dataset_name, baseline_results[dataset_name], adapter_results)
        del data
        gc.collect()

    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)
    for ds in ['compas', 'german', 'bank', 'lawschool', 'hospital']:
        if ds not in all_results:
            continue
        res = all_results[ds]
        base = baseline_results[ds]
        max_eo = max((abs(v) for k, v in res.items() if '_eo' in k), default=0)
        drop = base['acc'] - res['acc']
        status = ("SUCCESS" if max_eo <= 0.02 and drop <= 0.02
                  else "CLOSE" if max_eo <= 0.04 and drop <= 0.025
                  else "MISS")
        print(f"{ds.upper():12s}: Acc={res['acc']:.4f} (drop={drop:+.4f}) "
              f"EO={max_eo:.4f} [{status}]")


if __name__ == '__main__':
    main()

FAIRNESS PIPELINE: BBN PRIOR -> ADVERSARIAL ADAPTER -> GROUP THRESHOLDING
Device: cuda
Target: EO 0.01-0.02 | Accuracy drop <=2%

COMPAS
Cached COMPAS data to ./cache/preproc_compas.pkl
Training baseline MLP...

Training fair model...
  Stage 1: Task + group_balance + align_pos...
    Early stop at epoch 120
    Best val acc: 0.6810 | Stage 1 EO: 0.2057
  Stage 2: Adversarial adapter (lambda 0.05->0.3, tau=0.65)...
    Epoch   0: lambda=0.050  EO=0.1872  acc=0.6826
    Epoch   5: lambda=0.063  EO=0.1730  acc=0.6648
    Epoch  10: lambda=0.075  EO=0.2049  acc=0.6713
    Epoch  15: lambda=0.088  EO=0.2339  acc=0.6745
    Epoch  20: lambda=0.101  EO=0.1990  acc=0.6753
    Epoch  25: lambda=0.113  EO=0.1776  acc=0.6745
    Epoch  30: lambda=0.126  EO=0.2293  acc=0.6729
    Epoch  35: lambda=0.138  EO=0.2293  acc=0.6761
    Epoch  40: lambda=0.151  EO=0.1982  acc=0.6664
    Epoch  45: lambda=0.164  EO=0.2222  acc=0.6761
    Epoch  50: lambda=0.176  EO=0.2230  acc=0.6737
    Epoch  55: lambd

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'score_text': 'N', 'target': 'N', 'decile_score': 'N', 'nn_conf': 'N', 'race': 'N', 'c_charge_desc': 'N', 'priors_count': 'N', 'sex': 'N', 'nn_pred': 'N'}


    BBN modified 317/1235 (25.7%)
  Stage 3: Group-aware threshold calibration...
    tau_0=0.42, tau_1=0.50 -> EO=0.0005
    tau_0=0.70, tau_1=0.60 -> EO=0.0017
    Multi-attr best: max EO=0.1014

COMPAS RESULTS
Accuracy: Baseline=0.6980  Fair=0.6575  Drop=0.0405 (4.05%)
  Status: TOO HIGH

EO Metrics:
  race        : 0.2693 -> 0.1014  [MISS]
  sex         : 0.2377 -> 0.0051  [TARGET]

GERMAN
Cached German data to ./cache/preproc_german.pkl
Training baseline MLP...

Training fair model...
  Stage 1: Task + group_balance + align_pos...
    Early stop at epoch 250
    Best val acc: 0.7350 | Stage 1 EO: 0.0981
  Stage 2: Adversarial adapter (lambda 0.05->0.3, tau=0.4)...
    Epoch   0: lambda=0.050  EO=0.1217  acc=0.6900


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'status': 'N', 'nn_conf': 'N', 'housing': 'N', 'sex_label': 'N', 'target': 'N', 'nn_pred': 'N', 'amount': 'N'}


    Epoch   5: lambda=0.063  EO=0.0161  acc=0.7000
    EO <= 0.02 at epoch 5 - stopping.
    Adversary accuracy: 0.083 [OK]
  BBN calibration (weight=0.2)...
    BBN modified 12/200 (6.0%)
  Stage 3: Group-aware threshold calibration...
    tau_0=0.46, tau_1=0.54 -> EO=0.0000
    Stage 3.2: TPR curves cross - representation still biased.
    tau_0=0.35, tau_1=0.38 -> EO=0.0024
    Multi-attr best: max EO=0.0133

GERMAN RESULTS
Accuracy: Baseline=0.7250  Fair=0.7050  Drop=0.0200 (2.00%)
  Status: ACCEPTABLE

EO Metrics:
  age         : 0.1149 -> 0.3125  [MISS]
  sex         : 0.3208 -> 0.3369  [MISS]

BANK
Cached Bank data to ./cache/preproc_bank.pkl
Training baseline MLP...

Training fair model...
  Stage 1: Task + group_balance + align_pos...
    Best val acc: 0.8368 | Stage 1 EO: 0.0886
  Stage 2: Adversarial adapter (lambda 0.05->0.3, tau=0.5)...
    Epoch   0: lambda=0.050  EO=0.0584  acc=0.8164
    Epoch   5: lambda=0.066  EO=0.0513  acc=0.8158
    Epoch  10: lambda=0.082  EO=0.10

...

    BBN modified 76/1569 (4.8%)
  Stage 3: Group-aware threshold calibration...
    tau_0=0.71, tau_1=0.64 -> EO=0.0002
    Stage 3.2: TPR curves cross - representation still biased.
    tau_0=0.83, tau_1=0.86 -> EO=0.0006
    Multi-attr best: max EO=0.0333

BANK RESULTS
Accuracy: Baseline=0.8451  Fair=0.8260  Drop=0.0191 (1.91%)
  Status: ACCEPTABLE

EO Metrics:
  job         : 0.0826 -> 0.0333  [CLOSE]
  marital     : 0.0515 -> 0.0045  [TARGET]

LAWSCHOOL
Cached Law School data to ./cache/preproc_lawschool.pkl
Training baseline MLP...

Training fair model...
  Stage 1: Task + group_balance + align_pos...
    Best val acc: 0.8548 | Stage 1 EO: 0.1619
  Stage 2: Adversarial adapter (lambda 0.05->0.3, tau=0.5)...
    Epoch   0: lambda=0.050  EO=0.2631  acc=0.8812
    Epoch   5: lambda=0.066  EO=0.1701  acc=0.9285
    Epoch  10: lambda=0.082  EO=0.1279  acc=0.9357
    Epoch  15: lambda=0.097  EO=0.1160  acc=0.9386
    Epoch  20: lambda=0.113  EO=0.0776  acc=0.9455
    Epoch  25: lambda=0

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'nn_conf': 'N', 'race': 'N', 'bar': 'N', 'decile3': 'N', 'bar2': 'N', 'bar1': 'N', 'target': 'N', 'nn_pred': 'N', 'gender': 'N'}


    BBN modified 225/4478 (5.0%)
  Stage 3: Group-aware threshold calibration...
    tau_0=0.10, tau_1=0.10 -> EO=0.0000
    tau_0=0.10, tau_1=0.10 -> EO=0.0000
    Multi-attr best: max EO=0.0000

LAWSCHOOL RESULTS
Accuracy: Baseline=0.9495  Fair=0.9480  Drop=0.0016 (0.16%)
  Status: EXCELLENT

EO Metrics:
  gender      : 0.0286 -> 0.0000  [TARGET]
  race        : 0.2320 -> 0.0000  [TARGET]

HOSPITAL
Cached Hospital data to ./cache/preproc_hospital.pkl
Training baseline MLP...

Training fair model...
  Stage 1: Task + group_balance + align_pos...
    Best val acc: 0.6244 | Stage 1 EO: 0.0325
  Stage 2: Adversarial adapter (lambda 0.05->0.3, tau=0.45)...
    Epoch   0: lambda=0.050  EO=0.0362  acc=0.6063
    Epoch   5: lambda=0.063  EO=0.0379  acc=0.6245
    Epoch  10: lambda=0.075  EO=0.0307  acc=0.6140
    Epoch  15: lambda=0.088  EO=0.0359  acc=0.6103
    Epoch  20: lambda=0.101  EO=0.0436  acc=0.6037
    Epoch  25: lambda=0.113  EO=0.0311  acc=0.6151
    Epoch  30: lambda=0.126  EO=

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'age': 'N', 'nn_conf': 'N', 'race': 'N', 'num_medications': 'N', 'number_diagnoses': 'N', 'diabetesMed': 'N', 'target': 'N', 'nn_pred': 'N'}


    BBN modified 4325/10364 (41.7%)
  Stage 3: Group-aware threshold calibration...
    tau_0=0.66, tau_1=0.62 -> EO=0.0004
    tau_0=0.64, tau_1=0.65 -> EO=0.0002
    Multi-attr best: max EO=0.0191

HOSPITAL RESULTS
Accuracy: Baseline=0.6298  Fair=0.6153  Drop=0.0145 (1.45%)
  Status: GOOD

EO Metrics:
  race        : 0.0630 -> 0.0221  [CLOSE]
  sex         : 0.0250 -> 0.0191  [TARGET]

FINAL SUMMARY
COMPAS      : Acc=0.6575 (drop=+0.0405) EO=0.1014 [MISS]
GERMAN      : Acc=0.7050 (drop=+0.0200) EO=0.3369 [MISS]
BANK        : Acc=0.8260 (drop=+0.0191) EO=0.0333 [CLOSE]
LAWSCHOOL   : Acc=0.9480 (drop=+0.0016) EO=0.0000 [SUCCESS]
HOSPITAL    : Acc=0.6153 (drop=+0.0145) EO=0.0221 [CLOSE]
